# Experiment 7: Text Classification using Deep Learning Models

**Platform:** Google Colab  
**Objective:** Perform sentiment analysis on the IMDB Movie Reviews dataset using:
1. Logistic Regression
2. LSTM
3. Vanilla Transformer
4. Pretrained BERT

Compare the models using **Precision, Recall, F1-score, and Confusion Matrix**.

> Recommended: In Colab, select **Runtime → Change runtime type → T4 GPU** before running the neural-network and BERT sections. Run cells from top to bottom. The first dataset cell downloads the IMDB dataset through Hugging Face Datasets; an internet connection is required.

## 1. Install libraries and import packages

The notebook uses scikit-learn for Logistic Regression, TensorFlow/Keras for LSTM and the Vanilla Transformer, and Hugging Face Transformers with PyTorch for pretrained BERT.

In [ ]:
!pip -q install datasets transformers

In [ ]:
import os, re, random, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Load IMDB Movie Reviews dataset

The IMDB dataset contains movie reviews labelled as:
- `0` = Negative
- `1` = Positive

The original lab sheet links to the 50K IMDB reviews dataset. This notebook loads the same IMDB sentiment dataset using Hugging Face's dataset loader, which avoids manually configuring Kaggle API credentials in Colab.

In [ ]:
dataset = load_dataset("imdb")
train_df = pd.DataFrame(dataset["train"])[["text", "label"]]
test_df  = pd.DataFrame(dataset["test"])[["text", "label"]]

# Remove any empty reviews and ensure labels are integers
train_df = train_df.dropna().reset_index(drop=True)
test_df = test_df.dropna().reset_index(drop=True)
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))
display(train_df.head())
print("\\nClass distribution in training set:")
print(train_df["label"].value_counts())

### Optional: use smaller subsets for a faster Colab run

For a first run, the defaults below keep training practical. Set either value to `None` to use the full split. The held-out test subset is sampled separately and consistently across models.

In [ ]:
TRAIN_LIMIT = 12000   # set None for all 25,000 training reviews
TEST_LIMIT  = 5000    # set None for all 25,000 test reviews

if TRAIN_LIMIT is not None:
    train_df = train_df.sample(n=min(TRAIN_LIMIT, len(train_df)), random_state=SEED).reset_index(drop=True)
if TEST_LIMIT is not None:
    test_df = test_df.sample(n=min(TEST_LIMIT, len(test_df)), random_state=SEED).reset_index(drop=True)

X_train = train_df["text"].astype(str).tolist()
y_train = train_df["label"].to_numpy()
X_test = test_df["text"].astype(str).tolist()
y_test = test_df["label"].to_numpy()

print(f"Working train size: {len(X_train)} | test size: {len(X_test)}")

## 3. Evaluation helper

All models are evaluated on the same test split. For binary sentiment classification, positive class (`1`) is treated as the positive label.

In [ ]:
results = {}

def evaluate_model(model_name, y_true, y_pred):
    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }
    results[model_name] = metrics
    print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"], zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()
    return metrics

## 4. Model 1 — Logistic Regression

Convert reviews into TF-IDF numerical features, then train Logistic Regression. This is the traditional machine-learning baseline for comparison.

In [ ]:
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, C=2.0, random_state=SEED)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)

evaluate_model("Logistic Regression", y_test, lr_pred)

## 5. Prepare text for deep-learning models

Use Keras `TextVectorization` to convert text into padded integer sequences. The same vocabulary and sequence length are used for the LSTM and Vanilla Transformer.

In [ ]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 250
BATCH_SIZE = 64

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)
text_ds = tf.data.Dataset.from_tensor_slices(X_train).batch(256)
vectorizer.adapt(text_ds)

X_train_seq = vectorizer(tf.constant(X_train))
X_test_seq = vectorizer(tf.constant(X_test))
vocab_size = len(vectorizer.get_vocabulary())

print("Vocabulary size:", vocab_size)
print("Sequence tensor shape:", X_train_seq.shape)

## 6. Model 2 — LSTM

An LSTM (Long Short-Term Memory) network processes a review as a sequence of tokens and learns contextual information. The final dense layer with sigmoid activation predicts positive or negative sentiment.

In [ ]:
lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQUENCE_LENGTH,)),
    tf.keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
lstm_model.summary()

In [ ]:
lstm_history = lstm_model.fit(
    X_train_seq, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=BATCH_SIZE,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=1, restore_best_weights=True
    )],
    verbose=1
)

lstm_prob = lstm_model.predict(X_test_seq, batch_size=BATCH_SIZE).ravel()
lstm_pred = (lstm_prob >= 0.5).astype(int)
evaluate_model("LSTM", y_test, lstm_pred)

## 7. Model 3 — Vanilla Transformer

This is a small Transformer encoder built from scratch (not pretrained). It uses token embeddings, positional embeddings, multi-head self-attention, a feed-forward network, and global average pooling for classification.

In [ ]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads
        )
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim)
        ])
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = tf.keras.layers.Dropout(rate)
        self.drop2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs, training=training)
        x = self.norm1(inputs + self.drop1(attn_output, training=training))
        ffn_output = self.ffn(x, training=training)
        return self.norm2(x + self.drop2(ffn_output, training=training))


class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = tf.keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.pos_emb = tf.keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        return self.token_emb(x) + self.pos_emb(positions)


EMBED_DIM = 128
transformer_inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH,))
x = TokenAndPositionEmbedding(SEQUENCE_LENGTH, vocab_size, EMBED_DIM)(transformer_inputs)
x = TransformerBlock(EMBED_DIM, num_heads=4, ff_dim=256)(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
transformer_output = tf.keras.layers.Dense(1, activation="sigmoid")(x)

vanilla_transformer = tf.keras.Model(transformer_inputs, transformer_output)
vanilla_transformer.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)
vanilla_transformer.summary()

In [ ]:
transformer_history = vanilla_transformer.fit(
    X_train_seq, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=BATCH_SIZE,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=1, restore_best_weights=True
    )],
    verbose=1
)

transformer_prob = vanilla_transformer.predict(X_test_seq, batch_size=BATCH_SIZE).ravel()
transformer_pred = (transformer_prob >= 0.5).astype(int)
evaluate_model("Vanilla Transformer", y_test, transformer_pred)

## 8. Model 4 — Pretrained BERT

BERT (`bert-base-uncased`) is a pretrained Transformer encoder. It is fine-tuned for binary sentiment classification on the IMDB training reviews. This section uses Hugging Face Transformers and PyTorch.

**Runtime note:** BERT is resource-intensive. The defaults below use a subset for a Colab-friendly demonstration. Increase the limits or use the full dataset if your runtime has enough GPU memory and time.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

BERT_MODEL_NAME = "bert-base-uncased"
BERT_TRAIN_LIMIT = 4000  # set None to use all currently loaded training reviews
BERT_TEST_LIMIT = 2000   # set None to use all currently loaded test reviews
BERT_MAX_LENGTH = 192
BERT_EPOCHS = 2
BERT_BATCH_SIZE = 16

if BERT_TRAIN_LIMIT is not None:
    bert_train_texts, _, bert_train_labels, _ = train_test_split(
        X_train, y_train, train_size=min(BERT_TRAIN_LIMIT, len(X_train)),
        stratify=y_train, random_state=SEED
    )
else:
    bert_train_texts, bert_train_labels = X_train, y_train

if BERT_TEST_LIMIT is not None:
    bert_test_texts, _, bert_test_labels, _ = train_test_split(
        X_test, y_test, train_size=min(BERT_TEST_LIMIT, len(X_test)),
        stratify=y_test, random_state=SEED
    )
else:
    bert_test_texts, bert_test_labels = X_test, y_test

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

In [ ]:
class IMDBTorchDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


bert_train_dataset = IMDBTorchDataset(
    bert_train_texts, bert_train_labels, tokenizer, BERT_MAX_LENGTH
)
bert_test_dataset = IMDBTorchDataset(
    bert_test_texts, bert_test_labels, tokenizer, BERT_MAX_LENGTH
)

bert_train_loader = DataLoader(
    bert_train_dataset, batch_size=BERT_BATCH_SIZE, shuffle=True
)
bert_test_loader = DataLoader(
    bert_test_dataset, batch_size=BERT_BATCH_SIZE, shuffle=False
)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=2
).to(device)

optimizer = AdamW(bert_model.parameters(), lr=2e-5)

In [ ]:
for epoch in range(BERT_EPOCHS):
    bert_model.train()
    total_train_loss = 0.0

    for batch in bert_train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = bert_model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        total_train_loss += loss.item()

    print(f"Epoch {epoch+1}/{BERT_EPOCHS} | average training loss: "
          f"{total_train_loss / len(bert_train_loader):.4f}")

In [ ]:
bert_model.eval()
bert_predictions = []

with torch.no_grad():
    for batch in bert_test_loader:
        labels = batch["labels"]
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        logits = bert_model(**inputs).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        bert_predictions.extend(preds)

bert_pred = np.array(bert_predictions)
evaluate_model("Pretrained BERT", np.array(bert_test_labels), bert_pred)

## 9. Comparative analysis of all models

Create a single comparison table using the same evaluation metrics. Note that BERT may use a smaller subset of the test data as configured above, so its metric values are not a perfectly matched comparison unless `BERT_TEST_LIMIT = None`.

In [ ]:
comparison_df = pd.DataFrame(results.values()).set_index("Model")
display(comparison_df.round(4))

comparison_df[["Precision", "Recall", "F1-score"]].plot(
    kind="bar", figsize=(11, 5), ylim=(0, 1), rot=15
)
plt.title("Model Comparison: Precision, Recall and F1-score")
plt.ylabel("Score")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

best_model = comparison_df["F1-score"].idxmax()
print("Highest F1-score in this run:", best_model)

## 10. Conclusion and observations

Use the actual values printed by your Colab run in your practical record.

- **Logistic Regression:** A traditional machine-learning baseline. TF-IDF represents the review using weighted word and phrase frequencies.
- **LSTM:** Learns sequential patterns and contextual dependencies in review text.
- **Vanilla Transformer:** Uses self-attention to model relationships between tokens, trained from scratch on this dataset.
- **Pretrained BERT:** Starts with language representations learned during large-scale pretraining and is fine-tuned for sentiment classification.

Compare the Precision, Recall and F1-score, and inspect each confusion matrix to understand false-positive and false-negative predictions. The model with the highest F1-score in your run is the best according to that metric.

**Record the actual results:** Do not write assumed scores or claim a fixed winner. Scores depend on training subset, epochs, random seed, GPU/runtime, and preprocessing.

## 11. Viva quick revision

1. **What is sentiment analysis?** NLP task of classifying text as positive, negative, or another sentiment category.
2. **What is TF-IDF?** Term Frequency–Inverse Document Frequency; it weights terms based on their frequency in a document and rarity across documents.
3. **Why Logistic Regression?** It is an efficient supervised classification baseline.
4. **What is LSTM?** A recurrent neural network architecture designed to learn long-range sequence dependencies using gates and a cell state.
5. **What is a Transformer?** A neural architecture based on attention mechanisms rather than recurrence.
6. **What is self-attention?** A mechanism that lets each token weigh information from other tokens in the sequence.
7. **What is BERT?** Bidirectional Encoder Representations from Transformers, a pretrained bidirectional Transformer encoder.
8. **What is precision?** TP / (TP + FP).
9. **What is recall?** TP / (TP + FN).
10. **What is F1-score?** Harmonic mean of precision and recall: 2PR / (P + R).
11. **What is a confusion matrix?** A table of actual versus predicted class counts.
12. **Why compare models?** To study differences between a classical ML baseline, sequence learning, attention-based learning, and pretrained language representations.